[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/5-skills.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 5. Skills

A skill is a procedure file. It is not the system prompt, and it is not a second catalog.

`load_skill` reads `recommend` or the `gift-box` stub. The gift-box file tells the seller to call `query_catalog`. It does not list skus. The recommend file tells the seller to load Maya's preference and the allergen topic, then the catalog row.

A customer named Maya asks what jar to buy.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## The procedure is not the catalog

The gift-box stub is here so you can see a second procedure without a second price list.

In [ ]:
import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.memory
import tutorial.common.skills  # registers load_skill

from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.skills import load_skill

gift = load_skill({"name": "gift-box"})
print(gift)
check("query_catalog" in gift, "the gift-box stub points at the catalog")
check("HJ-" not in gift, "the skill is not a second catalog")

## Recommend for Maya

The skill names the steps. Preference, allergens, and the catalog still come from their own tools.

In [ ]:
reset_db()
result = run_agent("I'm Maya. What jar should I buy?", system_text())
print("ANSWER:", result["text"])
log = "\n".join(result["tool_log"])
lower = result["text"].lower()
offered_sesame = "sesame crunch" in lower and not any(
    word in lower for word in ("not", "avoid", "skip", "instead")
)
check("load_skill" in log, "the recommend procedure was loaded")
check("get_preference" in log, "Maya's preference was loaded")
check("query_catalog" in log or "get_store_fact" in log, "the answer uses store data, not only the skill file")
check("chili" in lower, "the recommendation fits Maya's favorite")
check(not offered_sesame, "sesame crunch is not the jar being sold")